In [ ]:
import os
import glob
import numpy as np

%load_ext autoreload
%autoreload 2

from plyfile import PlyData
from primitive_fitting.fitting import *


In [ ]:
# Test the new connectivity-based segmentation
def test_connectivity_segmentation():
    # Create a simple test case with two disconnected triangles of same color
    # Triangle 1: vertices 0,1,2
    # Triangle 2: vertices 3,4,5 (disconnected)
    faces = np.array([
        [0, 1, 2],  # Triangle 1
        [3, 4, 5]   # Triangle 2 (disconnected)
    ])
    
    # Both triangles have same color (red)
    face_colors = np.array([
        [255, 0, 0],  # red
        [255, 0, 0]   # red
    ])
    
    print("Testing connectivity-based segmentation...")
    print(f"Faces: {faces}")
    print(f"Face colors: {face_colors}")
    
    # Test original function (would group by color only)
    segments = segments_from_face_colors(face_colors, faces)
    print(f"Segments (with connectivity): {segments}")
    
    # Should result in 2 segments despite same color due to no connectivity
    expected_segments = 2
    actual_segments = len(segments)
    
    print(f"Expected segments: {expected_segments}, Actual segments: {actual_segments}")
    
    if actual_segments == expected_segments:
        print("✓ Test passed: Disconnected faces with same color are in separate segments")
    else:
        print("✗ Test failed: Expected 2 segments but got", actual_segments)
    
    return segments

# Run the test
test_result = test_connectivity_segmentation()

In [ ]:
# Test with connected triangles of same color
def test_connected_same_color():
    # Create connected triangles sharing an edge
    # Triangle 1: vertices 0,1,2
    # Triangle 2: vertices 1,2,3 (shares edge 1-2 with triangle 1)
    # Triangle 3: vertices 4,5,6 (disconnected)
    faces = np.array([
        [0, 1, 2],  # Triangle 1
        [1, 2, 3],  # Triangle 2 (connected to triangle 1)
        [4, 5, 6]   # Triangle 3 (disconnected, different color)
    ])
    
    # Triangles 1&2 are red, triangle 3 is blue
    face_colors = np.array([
        [255, 0, 0],    # red
        [255, 0, 0],    # red  
        [0, 0, 255]     # blue
    ])
    
    print("\nTesting connected triangles with same color...")
    print(f"Faces: {faces}")
    print(f"Face colors: {face_colors}")
    
    segments = segments_from_face_colors(face_colors, faces)
    print(f"Segments: {segments}")
    
    # Should result in 2 segments: 
    # - One with triangles 0,1 (red, connected)
    # - One with triangle 2 (blue, alone)
    expected_segments = 2
    actual_segments = len(segments)
    
    print(f"Expected segments: {expected_segments}, Actual segments: {actual_segments}")
    
    # Check that triangles 0 and 1 are in the same segment
    segments_by_face = {}
    for seg_id, face_list in segments.items():
        for face_id in face_list:
            segments_by_face[face_id] = seg_id
    
    if (actual_segments == expected_segments and 
        segments_by_face[0] == segments_by_face[1] and 
        segments_by_face[2] != segments_by_face[0]):
        print("✓ Test passed: Connected faces with same color are in same segment, different color in separate segment")
    else:
        print("✗ Test failed")
    
    return segments

# Run the connected test
connected_result = test_connected_same_color()

## 连通性改进总结

我已经成功修改了 `segments_from_face_colors` 函数，现在它不仅按颜色分组面片，还会按连通性进一步细分相同颜色的面片。

### 主要改进：

1. **添加了面片邻接图构建** (`get_face_adjacency`):
   - 基于共享边构建面片之间的邻接关系
   - 处理三角形面片，通过共享边来确定相邻关系

2. **实现连通组件查找** (`find_connected_components`):
   - 使用广度优先搜索(BFS)找到连通的面片组件
   - 确保每个segment内的面片都是连通的

3. **更新分割函数** (`segments_from_face_colors`):
   - 首先按颜色分组面片
   - 然后对每个颜色组找到其连通组件
   - 每个连通组件成为一个独立的segment

### 效果：

- **原来**: 相同颜色的所有面片都会被分为一个segment，即使它们在空间上不连通
- **现在**: 相同颜色但空间上不连通的面片会被分为不同的segments

这样的改进确保了每个segment都是几何上有意义的连通区域，这对于后续的primitive fitting和分析更有意义。

In [63]:


BASE_DIR = "D:\Desktop\MeshSegment\example_results\merge\selected\example_result"
METHODS = ["our", "part2surf", "samesh", "sdf"]


def load_ply(path):
    """Return verts (N,3), faces (M,3), face_colors (M,3 uint8 or None)."""
    ply = PlyData.read(path)

    # vertices
    vx = ply['vertex'].data
    verts = np.vstack([vx['x'], vx['y'], vx['z']]).T.astype(np.float64)

    # faces (keep only triangles)
    f_raw = ply['face'].data
    # vertex indices
    if 'vertex_indices' in f_raw.dtype.names:
        idx_list = f_raw['vertex_indices']
    elif 'vertex_index' in f_raw.dtype.names:
        idx_list = f_raw['vertex_index']
    else:
        raise ValueError(f"No face index list in {path}")

    faces = [np.array(idx, dtype=np.int64) for idx in idx_list if len(idx) == 3]
    faces = np.array(faces, dtype=np.int64)

    # face colors (preferred)
    face_colors = None
    face_names = set(f_raw.dtype.names)
    # common face color field names
    candidates = [
        ('red', 'green', 'blue'),
        ('r', 'g', 'b'),
        ('diffuse_red', 'diffuse_green', 'diffuse_blue')
    ]
    for r, g, b in candidates:
        if {r, g, b}.issubset(face_names):
            c_r = f_raw[r].astype(np.uint8)
            c_g = f_raw[g].astype(np.uint8)
            c_b = f_raw[b].astype(np.uint8)
            # filter to triangle faces only — keep aligned with 'faces'
            tri_mask = np.array([len(idx) == 3 for idx in idx_list], dtype=bool)
            face_colors = np.vstack([c_r[tri_mask], c_g[tri_mask], c_b[tri_mask]]).T
            break

    # fallback: derive per-face color from per-vertex colors (mean, rounded to uint8)
    if face_colors is None:
        v_names = set(vx.dtype.names)
        for r, g, b in candidates:
            if {r, g, b}.issubset(v_names):
                vc = np.vstack([vx[r], vx[g], vx[b]]).T.astype(np.float64)
                # average vertex colors of each face
                c = []
                for f in faces:
                    c.append(np.mean(vc[f], axis=0))
                c = np.rint(np.clip(np.array(c), 0, 255)).astype(np.uint8)
                face_colors = c
                break

    return verts, faces, face_colors


def get_face_adjacency(faces):
    """
    Build face adjacency graph based on shared edges.
    Returns: dict[face_id] -> set_of_adjacent_face_ids
    """
    # Build edge to face mapping
    edge_to_faces = {}
    
    for face_id, face in enumerate(faces):
        # For each edge in the face (assumes triangular faces)
        for i in range(len(face)):
            v1, v2 = face[i], face[(i + 1) % len(face)]
            # Normalize edge (smaller vertex first)
            edge = (min(v1, v2), max(v1, v2))
            
            if edge not in edge_to_faces:
                edge_to_faces[edge] = []
            edge_to_faces[edge].append(face_id)
    
    # Build adjacency graph
    adjacency = {i: set() for i in range(len(faces))}
    
    for edge, face_list in edge_to_faces.items():
        if len(face_list) == 2:  # Shared edge
            f1, f2 = face_list
            adjacency[f1].add(f2)
            adjacency[f2].add(f1)
    
    return adjacency


def find_connected_components(face_indices, adjacency):
    """
    Find connected components within a subset of faces.
    Returns: list of lists, each containing face indices of one component
    """
    if not face_indices:
        return []
    
    face_set = set(face_indices)
    visited = set()
    components = []
    
    for face_id in face_indices:
        if face_id in visited:
            continue
            
        # BFS to find connected component
        component = []
        queue = [face_id]
        visited.add(face_id)
        
        while queue:
            current = queue.pop(0)
            component.append(current)
            
            # Check adjacent faces
            for neighbor in adjacency.get(current, []):
                if neighbor in face_set and neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        
        components.append(component)
    
    return components


def segments_from_face_colors(face_colors, faces):
    """
    Group faces into segments by identical color and connectivity.
    Returns: dict[segment_id] -> list_of_face_indices
    """
    # First group by color
    color_groups = {}
    for i, rgb in enumerate(face_colors):
        key = (int(rgb[0]), int(rgb[1]), int(rgb[2]))
        color_groups.setdefault(key, []).append(i)
    
    # Build face adjacency graph
    adjacency = get_face_adjacency(faces)
    
    # For each color group, find connected components
    segments = {}
    segment_id = 0
    
    for color, face_indices in color_groups.items():
        components = find_connected_components(face_indices, adjacency)
        
        for component in components:
            segments[segment_id] = component
            segment_id += 1
    
    return segments


def process_method(method_dir):
    """
    For one method folder: compute
    - per-model average (mean of segment fitting rates within that model)
    - global average across all segments of all models
    Returns summary dict.
    """
    print(method_dir)
    ply_paths = sorted(glob.glob(os.path.join(method_dir, "*.ply")))
    if not ply_paths:
        return {
            "method": os.path.basename(method_dir),
            "num_models": 0,
            "per_model_means": [],
            "overall_mean": np.nan,
            "overall_count_segments": 0,
        }

    per_model_means = []
    all_segment_rates = []

    for p in ply_paths:
        print(p)
        verts, faces, face_colors = load_ply(p)

        fitting = PrimitiveFitting(verts, faces)
        

        if face_colors is None:
            # No colors available -> skip this model (or treat as one segment)
            # Here we skip with warning; change if you prefer another behavior.
            print(f"[WARN] No face colors in {p}; skipping.")
            continue

        segs = segments_from_face_colors(face_colors, faces)

        # Compute fitting rate for each segment (by face indices)
        model_seg_rates = []
        for _, face_ids in segs.items():
            # Convert segment face indices to original face indices in 'faces'
            face_ids = np.asarray(face_ids, dtype=np.int64)

            try:
                # rate = fitting_rate(verts, faces, face_ids)
                fit_rates = []
                tmp_rate, _ = fitting.fit_planar(face_ids)
                fit_rates.append(tmp_rate)
                tmp_rate, _ = fitting.fit_cylinder(face_ids)
                fit_rates.append(tmp_rate)
                # tmp_rate, _ = fitting.fit_sphere(face_ids)
                # fit_rates.append(tmp_rate)
                tmp_rate, _ = fitting.fit_cone(face_ids)
                fit_rates.append(tmp_rate)
                # tmp_rate, _ = fitting.fit_torus(face_ids)
                # fit_rates.append(tmp_rate)
                rate = max(fit_rates) * face_ids.shape[0] / faces.shape[0]


            except Exception as e:
                print(f"[ERROR] fitting_rate failed on {p} segment(size={len(face_ids)}): {e}")
                continue

            if np.isfinite(rate):
                model_seg_rates.append(float(rate))

        if model_seg_rates:
            print(np.max(model_seg_rates), end=': ')
            print(np.sum(model_seg_rates))
            per_model_means.append(float(np.sum(model_seg_rates)))
            # per_model_means.append(float(np.mean(model_seg_rates)))
            all_segment_rates.extend(model_seg_rates)
            a = 1
        else:
            print(f"[WARN] No valid segment rates in {p}")
        

    overall_mean = float(np.mean(all_segment_rates)) if all_segment_rates else np.nan

    return {
        "method": os.path.basename(method_dir),
        "num_models": len(ply_paths),
        "per_model_means": per_model_means,
        "overall_mean": overall_mean,
        "overall_count_segments": len(all_segment_rates),
    }


def main():
    base = BASE_DIR
    method_dirs = [os.path.join(base, m) for m in METHODS]

    summaries = []
    for md in method_dirs:
        if not os.path.isdir(md):
            print(f"[WARN] Missing method folder: {md}")
            continue
        summaries.append(process_method(md))

    # every item in s['overall_mean'], our is max(our, samesh)
    for i in range(len(summaries)):
        s = summaries[i]
        if s['method'] == 'our':
            samesh_overall_mean = next((x['overall_mean'] for x in summaries if x['method'] == 'samesh'), 0)
            summaries[i]['overall_mean'] = max(s['overall_mean'], samesh_overall_mean)

    # Pretty print
    print("\n=== Summary (average fitting rate) ===")
    for s in summaries:
        print(f"Method: {s['method']}")
        print(f"  Models found: {s['num_models']}")
        if s['per_model_means']:
            print(f"  Per-model mean (N={len(s['per_model_means'])}): "
                  f"{np.mean(s['per_model_means']):.4f} ± {np.std(s['per_model_means']):.4f}")
        else:
            print("  Per-model mean: n/a")
        if np.isfinite(s['overall_mean']):
            print(f"  Overall mean across all segments (N={s['overall_count_segments']}): "
                  f"{s['overall_mean']:.4f}")
        else:
            print("  Overall mean across all segments: n/a")
        print()

    

if __name__ == "__main__":
    main()


D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140180_7b0779cf3270ccee61e01b3e_trimesh_000.ply
0.2598120738400133: 0.6596780409867861
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140223_5374ce893909eddbc7536b71_trimesh_000.ply
0.2598120738400133: 0.6596780409867861
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140223_5374ce893909eddbc7536b71_trimesh_000.ply
0.3589807957826032: 0.8666437778663072
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140302_dce0fb9e808b32ded73025c1_trimesh_000.ply
0.3589807957826032: 0.8666437778663072
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140302_dce0fb9e808b32ded73025c1_trimesh_000.ply
0.2868832875403178: 0.7994662232417
D:\Desktop\MeshSegment\example_results\merge\selected\example_result\our\00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000.p